In [2]:
!pip install spacy
!python -m spacy download en_core_web_sm

  Using cached spacy-3.8.14-cp312-cp312-win_amd64.whl.metadata (28 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.15-cp312-cp312-win_amd64.whl.metadata (2.3 kB)
  Using cached cymem-2.0.13-cp312-cp312-win_amd64.whl.metadata (9.9 kB)
  Using cached preshed-3.0.13-cp312-cp312-win_amd64.whl.metadata (5.4 kB)
  Using cached thinc-8.3.13-cp312-cp312-win_amd64.whl.metadata (15 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.5.3-cp312-cp312-win_amd64.whl.metadata (20 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-1.0.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached confection-1.3.3-py3-none-any.whl.metadata (19 kB)
  Using cached blis-1.3.3-cp312-cp312-win_amd64.whl.metadata (7.7 kB)
Using cached spacy-3.8.14-cp312-cp312-win_amd64.whl (14.2 MB)
Using cached catalogue-2.0.10-py3


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------- ------------------------------ 3.1/12.8 MB 16.8 MB/s eta 0:00:01
     ------------ --------------------------- 3.9/12.8 MB 16.8 MB/s eta 0:00:01
     ---------------- ----------------------- 5.2/12.8 MB 8.6 MB/s eta 0:00:01
     ------------------ --------------------- 6.0/12.8 MB 7.7 MB/s eta 0:00:01
     --------------------- ------------------ 6.8/12.8 MB 6.8 MB/s eta 0:00:01
     ----------------------- ---------------- 7.6/12.8 MB 6.2 MB/s eta 0:00:01
     ------------------------- -------------- 8.1/12.8 MB 5.5 MB/s eta 0:00:01
     ---------------------------- ----------- 9.2/12.8 MB 5.5 MB/s eta 0:00:01
     --------------------------------- ------ 10.7/12.8 MB 5.7 MB/s eta 0:00:01
     ------------------------------------ --- 11.8/12.8 MB 5.7 MB/s eta 0:00:01
     ---------------------------------------  12.6/12.8 MB 5.5 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import spacy
nlp = spacy.load("en_core_web_sm")

test = "Great cooler excellent air flow and price is amazing"
doc = nlp(test)

for token in doc:
    print(f"{token.text:15} → {token.pos_}")

Great           → ADJ
cooler          → ADJ
excellent       → ADJ
air             → NOUN
flow            → NOUN
and             → CCONJ
price           → NOUN
is              → AUX
amazing         → ADJ


In [4]:
from collections import Counter

def summarize_reviews(reviews: list) -> dict:
    if not reviews:
        return {"summary": "No reviews found.", "keywords": []}

    combined = " ".join(reviews)
    doc = nlp(combined)

    # Sirf NOUN aur ADJ nikalo — stopwords ignore
    keywords = [
        token.text.lower() for token in doc
        if token.pos_ in ['NOUN', 'ADJ']
        and not token.is_stop
        and len(token.text) > 2
    ]

    # Top 5 keywords
    top_keywords = Counter(keywords).most_common(5)

    # Summary sentence banao
    keyword_str = ', '.join([k for k, v in top_keywords])
    summary = f"Customers mainly talked about: {keyword_str}"

    return {
        "summary": summary,
        "keywords": top_keywords
    }

In [7]:
result = summarize_reviews(sample_reviews)
print("Summary:", result['summary'])
print("\nTop Keywords:")
for word, count in result['keywords']:
    print(f"  {word}: {count} times")

Summary: Customers mainly talked about: good, cooler, product, air, flow

Top Keywords:
  good: 6 times
  cooler: 5 times
  product: 5 times
  air: 4 times
  flow: 2 times


In [6]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Jatin\OneDrive\Desktop\DL projects\ecom-ai\data\reviews_clean.csv", encoding="latin1")

sample_reviews = df['Summary'].dropna().head(15).tolist()
sample_ratings = df['Rate'].dropna().head(15).tolist()

print(f"Reviews loaded: {len(sample_reviews)}")
for r in sample_reviews:
    print(f"- {r}")

Reviews loaded: 15
- Great cooler.. excellent air flow and for this price. It's so amazing and unbelievable.Just love it ÂÂ?Â?Â
- Best budget 2 fit cooler. Nice cooling
- The quality is good but the power of air is decent
- Very bad product it's a only a fan
- Ok ok product
- The cooler is really fantastic and provides good air flow. Highly recommended
- Very good product
- Very nice
- Very bad cooler
- Very good
- Beautiful product good material and perfectly working
- Awesome
- Good
- Wonderful product, Must buy
- Nice air cooler, smart cool breeze producer


In [8]:
custom_stopwords = ['good', 'product', 'nice', 'best', 'great', 'bad', 
                    'very', 'really', 'just', 'also', 'much', 'well']

def summarize_reviews(reviews: list) -> dict:
    if not reviews:
        return {"summary": "No reviews found.", "keywords": []}

    combined = " ".join(reviews)
    doc = nlp(combined)

    keywords = [
        token.text.lower() for token in doc
        if token.pos_ in ['NOUN', 'ADJ']
        and not token.is_stop
        and len(token.text) > 2
        and token.text.lower() not in custom_stopwords
    ]

    top_keywords = Counter(keywords).most_common(5)
    keyword_str = ', '.join([k for k, v in top_keywords])
    summary = f"Customers mainly talked about: {keyword_str}"

    return {
        "summary": summary,
        "keywords": top_keywords
    }

# Test
result = summarize_reviews(sample_reviews)
print("Summary:", result['summary'])
print("\nTop Keywords:")
for word, count in result['keywords']:
    print(f"  {word}: {count} times")

Summary: Customers mainly talked about: cooler, air, flow, excellent, price

Top Keywords:
  cooler: 5 times
  air: 4 times
  flow: 2 times
  excellent: 1 times
  price: 1 times
